# ✨Evaluating spectral reflectance separability✨

This notebook can be used to evaluate spectral separability of point-source spectra using three methods: Hierarchical Cluster Analysis, Spectral Angle Classification, and Random Forest Classification.

## 🔨 Set-up 
Importing modules and data

In [173]:
# Import required scripts
import numpy as np
import pvclust
import speca.preprocessing as prep
import speca.visualizing as sv
import speca.auxiliary as aux
import speca.spectral_angle_method as sam
import speca.random_forest as rfc
import speca.pca_decomposition as pca

In [ ]:
# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

##  🏭 Preprocessing

This section runs the Preprocessor class which imports the data labelled by class name, cleans the data, and filters to desired classes. If a scaler is chosen, it will scale the data accordingly.

In [164]:
# Define preprocessing parameters
path = r"C:\Users\s4770224\Documents\coding\Spectral_analysis\Combined_analysis\MEL_NZ_TAS_spectra.csv"
bad_bands_list = list(map(str,(range(753, 769))))
scaler = "standard"
target_scheme = "all_macros"
target_sites = None
target_setup = None
start_nm = 460
end_nm = 925

In [ ]:
#Run the preprocessing chain
prepper = prep.PreprocessRefl(data_path = path,
                              bad_bands = bad_bands_list,
                              scaler = scaler,
                              target_scheme = target_scheme,
                              target_setup = target_setup,
                              target_sites= target_sites,
                              start_nm = start_nm,
                              end_nm = end_nm)
prepper.default_chain()
spectra_prepped = prepper.spectra_norm
labels_prepped = prepper.labels_filtered

In [ ]:
labels_prepped.tail()


## 📈 Visualise

This section runs the plotting class using the already defined parameters from preprocessing. Spectra to plot are defined by the user by target types and collection site.

In [171]:
# Initialize visualizer with existing parameters
vizer = sv.Visualizer(spectra_prepped, 
                      labels_prepped, 
                      start_nm, 
                      end_nm)

# Calculate class statistics for plotting
vizer.class_stats(grouping="Class", 
                  plot_by_site = True)

### Chose spectra to plot

Search words below defines what target types you want to plot. Search sites is optional, but will let you chose to plot spectra from only the specified site(s). The mode parameter can be `any` or `all` to indicate if all of the search terms need to be met or if all must be found to be returned.

In [172]:
# Define search terms
search_words = ['acrocarpia', 'carpophyllum', 'cystophora', 'durvillaea', 'ecklonia','filamentous_rhodophyte', 'frondose_rhodophyte', 'hormosira',
       'macrocystis', 'phyllospora', 'sargassum', 'scytosiphon', 'ulva', 'petalonia', 'undaria']
search_sites = []
mode = any

In [ ]:
# Run the plotter
vizer.plot_search_terms(search_words= search_words, mode = any, search_sites = search_sites)

##  🔺Hierarchical Cluster Analysis 🔺
This section uses the pvclust adaptation for Python to run an HCA on the prepared spectra. An optional class means dataset is also calculated for clustering means. 

In [146]:
# Calculate class means for clustering
class_means = aux.calc_class_averages(spectra_prepped,
                                      labels_prepped,
                                      label_col="Class")

In [ ]:
# Chose data to cluster
clust_data = class_means

# Perform clustering
pv = pvclust.PvClust(clust_data.T, method = "ward", metric = "euclidean", nboot = 10000, parallel = True)

In [ ]:
# Plot dendrogram
pv.plot(labels= clust_data.index.to_list())

In [ ]:
# Plot clustering standard errors
pv.seplot(pvalue= "AU", annotate=True)

## 📐 Spectral Angle Method
This section runs an adaptation of the conventional spectral angle mapper to be applied to point-source reflectance data.

In [174]:
# strat an instance fo the SAM class
sa = sam.SAM(spectra_prepped, labels_prepped, "Class")

In [ ]:
# Run the chain module to perform all SAM steps
sa.sam_chain()

## 🌲🌳 Random Forest 🌳🌲

This section implements a random forest classifier on the spectra. It includes:
- PCA decomposition
- N-comps determination for PCA
- Hyperparameter grid search
- Random Forest classification

### Extract Principal Components

In [176]:
components, ev_ratio, ev = pca.run_pca(spectra_prepped, n_comps=20)

### Determine number of components to keep

In [ ]:
# Check recommended components with elbow and kaiser methods
elbow = pca.find_elbow(ev_ratio)
print(f"Elbow found at: {elbow} components \n  Explained variance ratio: {ev_ratio[elbow]} \n  Total explained variance: {np.sum(ev_ratio[:elbow+1])}")

kaiser = pca.find_kaiser(ev)
print(f"Kaiser value found at: {kaiser} components \n  Explained variance ratio: {ev_ratio[kaiser]} \n  Total explained variance: {np.sum(ev_ratio[:kaiser+1])}")

In [6]:
# Check number of components with accuracy based methods
# Run  a set of RFCs to use for accuracy estimates
accub = pca.NCompsByAccuracy(components, labels_prepped, labels_col = "Class")
accub.calc_mean_accuracies(max_n = 20)

In [ ]:
# Evaluate proximity and marginal gains from RFC runs
proximity = accub.find_n_by_max_proximity()
margin = accub.find_n_by_margin()

# View results
print(proximity)
print(margin)

You now have four suggestions for the number of components to use in subsequent analysis:
- Two based on dataset variance (elbow, kaiser) 
- Two based on preliminary classification accuracy (margin, proximity). 

The best suited option is subjective and will affect:
- data volume 💼
- computational requirements 💻 
- processing time ⏳
- and accuracy 📍

### Perform the Random Forest Classification

In [180]:
# Initialize a classifer with your chosen number of components
forest = rfc.RandoForest(spectra_prepped, labels_prepped, n_comps = 7, labels_col= "Class", runs = 1000)

In [ ]:
# Run a grid search

# OPTIONAL: Define the parameters for the grid search, otherwise uses defaults
params_dt = {'min_samples_split': [2,3,4],
'n_estimators': [55, 75, 95, 115, 200, 500],
'max_depth': [10, 20, 30, 40, 50, None],
'max_leaf_nodes': [40, 50, 60, None]
}

# Perform search and view results
forest.grid_search(parameters=params_dt)

In [ ]:
# Calculate aggregate results over many random forest runs
forest.hyperparams = {
    "n_estimators": 95,
    "max_depth": 25,
    "min_samples_split": 2,
    "max_leaf_nodes": 50,
    "bootstrap": True
}
forest.many_rfc_runs()

### Plot the RFC confusion matrix

In [ ]:
forest.plot_cm()